# 03 - MACE Training
Train the message-passing MACE model with the same training setup as ACE and model-specific architecture choices.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load static datasets ONCE before the loop to save time
print('Pre-loading static datasets...')
val_ds = MDTrajectoryDataset('../data/val.extxyz', cutoff=5.0)
test_ds = MDTrajectoryDataset('../data/test.extxyz', cutoff=5.0)

import time

fractions = [10, 40, 70, 100]
metrics_dict = {}

for frac in fractions:
    print(f"\n{'='*40}")
    print(f"Starting MACE Training for {frac}% data fraction")
    print(f"{'='*40}\n")
    
    # 1. Load Data
    train_path = f"../data/train_{frac}.extxyz"
    train_ds = MDTrajectoryDataset(train_path, cutoff=5.0)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

    # 2. Re-Initialize MACE model (resets weights)
    model = MACEWrapper(
        num_elements=120,
        r_cut=5.0,
        num_radial=8,
        l_max=2,
        num_blocks=2,
        node_dim=16
    )

    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    trainer = BenchmarkTrainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=val_loader,
        device="cuda" if torch.cuda.is_available() else "cpu",
        energy_weight=1.0,
        force_weight=100.0
    )
    
    # 3. Train
    metrics_df = trainer.train(max_epochs=30, patience=10)
    metrics_df.to_csv(f"../data/mace_metrics_{frac}.csv", index=False)
    
    # 4. Test (In-Distribution)
    test_metrics = trainer.test_epoch(test_loader)
    pd.DataFrame([test_metrics]).to_csv(f"../data/mace_test_metrics_{frac}.csv", index=False)
    
    
    # 6. Save model
    torch.save(model.state_dict(), f"../data/mace_model_{frac}.pth")
    
    metrics_dict[frac] = metrics_df
    
    # 7. Free GPU Memory
    del model, optimizer, trainer, train_loader, val_loader, test_loader
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Pre-loading static datasets...
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).

Starting MACE Training for 10% data fraction

Pre-computing graphs for 100 structures (cutoff=5.0 Å)...


...

  Done. Dataset ready (400 graphs).
Epoch 000 | Time: 7.16s | Train E MAE: 2271.16 meV/atom | Train F MAE: 213.47 meV/Å | Val E MAE: 270.19 meV/atom | Val F MAE: 161.17 meV/Å
Epoch 001 | Time: 6.60s | Train E MAE: 242.08 meV/atom | Train F MAE: 85.85 meV/Å | Val E MAE: 208.50 meV/atom | Val F MAE: 35.36 meV/Å
Epoch 002 | Time: 6.30s | Train E MAE: 146.48 meV/atom | Train F MAE: 30.55 meV/Å | Val E MAE: 11.38 meV/atom | Val F MAE: 13.55 meV/Å
Epoch 003 | Time: 6.31s | Train E MAE: 66.68 meV/atom | Train F MAE: 13.83 meV/Å | Val E MAE: 79.08 meV/atom | Val F MAE: 14.30 meV/Å
Epoch 004 | Time: 6.31s | Train E MAE: 42.96 meV/atom | Train F MAE: 12.82 meV/Å | Val E MAE: 19.59 meV/atom | Val F MAE: 10.18 meV/Å
Epoch 005 | Time: 6.30s | Train E MAE: 21.20 meV/atom | Train F MAE: 9.52 meV/Å | Val E MAE: 13.90 meV/atom | Val F MAE: 8.25 meV/Å
Epoch 006 | Time: 6.31s | Train E MAE: 10.88 meV/atom | Train F MAE: 8.39 meV/Å | Val E MAE: 12.11 meV/atom | Val F MAE: 7.61 meV/Å
Epoch 007 | Time: 6.30

Building graphs:   0%|          | 0/700 [00:00<?, ?it/s]

  Done. Dataset ready (700 graphs).
Epoch 000 | Time: 11.74s | Train E MAE: 1274.48 meV/atom | Train F MAE: 154.52 meV/Å | Val E MAE: 310.69 meV/atom | Val F MAE: 84.32 meV/Å
Epoch 001 | Time: 11.03s | Train E MAE: 104.40 meV/atom | Train F MAE: 85.37 meV/Å | Val E MAE: 0.49 meV/atom | Val F MAE: 56.47 meV/Å
Epoch 002 | Time: 11.04s | Train E MAE: 4.00 meV/atom | Train F MAE: 49.53 meV/Å | Val E MAE: 3.45 meV/atom | Val F MAE: 35.64 meV/Å
Epoch 003 | Time: 11.11s | Train E MAE: 1.16 meV/atom | Train F MAE: 21.62 meV/Å | Val E MAE: 1.57 meV/atom | Val F MAE: 9.85 meV/Å
Epoch 004 | Time: 11.19s | Train E MAE: 0.45 meV/atom | Train F MAE: 7.85 meV/Å | Val E MAE: 0.61 meV/atom | Val F MAE: 6.36 meV/Å
Epoch 005 | Time: 11.18s | Train E MAE: 0.32 meV/atom | Train F MAE: 5.76 meV/Å | Val E MAE: 0.12 meV/atom | Val F MAE: 5.17 meV/Å
Epoch 006 | Time: 11.24s | Train E MAE: 0.17 meV/atom | Train F MAE: 4.97 meV/Å | Val E MAE: 0.13 meV/atom | Val F MAE: 4.75 meV/Å
Epoch 007 | Time: 11.24s | Train

Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Epoch 000 | Time: 16.57s | Train E MAE: 198.08 meV/atom | Train F MAE: 123.69 meV/Å | Val E MAE: 16.21 meV/atom | Val F MAE: 65.95 meV/Å
Epoch 001 | Time: 15.83s | Train E MAE: 11.21 meV/atom | Train F MAE: 27.62 meV/Å | Val E MAE: 5.66 meV/atom | Val F MAE: 8.03 meV/Å
Epoch 002 | Time: 15.95s | Train E MAE: 2.59 meV/atom | Train F MAE: 6.83 meV/Å | Val E MAE: 0.53 meV/atom | Val F MAE: 4.17 meV/Å
Epoch 003 | Time: 16.02s | Train E MAE: 0.38 meV/atom | Train F MAE: 3.69 meV/Å | Val E MAE: 0.22 meV/atom | Val F MAE: 3.22 meV/Å
Epoch 004 | Time: 16.10s | Train E MAE: 0.51 meV/atom | Train F MAE: 3.00 meV/Å | Val E MAE: 1.04 meV/atom | Val F MAE: 2.87 meV/Å
Epoch 005 | Time: 16.08s | Train E MAE: 0.39 meV/atom | Train F MAE: 2.75 meV/Å | Val E MAE: 0.43 meV/atom | Val F MAE: 2.51 meV/Å
Epoch 006 | Time: 16.17s | Train E MAE: 0.38 meV/atom | Train F MAE: 2.59 meV/Å | Val E MAE: 0.74 meV/atom | Val F MAE: 2.69 meV/Å
Epoch 007 | Time: 16.18s | Train E MAE

In [3]:
model = MACEWrapper(num_elements=120, r_cut=5.0, num_radial=8, l_max=2, num_blocks=2, node_dim=16)
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 417,825
Trainable Parameters: 417,825
